# União de tabelas similares
Monta uma única tabela com todas as tablas com o mesmo padrão de nome.  
Inclui todas as colunas de todas as tabelas e as que não tiverem alguma coluna que outra tenha, fica com o valor NULL.

In [0]:
%run "/Workspace/Jurídico/funcao_tratamento_fechamento/common_functions"

In [0]:
from pyspark.sql.types import DoubleType, LongType, TimestampType, StringType

In [0]:
# Definir o schema e o regex do nome das tabelas
nome_tabelas = "'tb_fechamento_civel%'"
nome_schema = "databox.juridico_comum"
nome_tabela_unida = "tb_fech_civel_provisao"

### 1 - Definição das funções

In [0]:
# Carrega as tabelas em DF e retorna uma lista de DFs
def load_dataframes(schema, table_names):
    """
    Carrega as tabelas em DF e retorna uma lista de DFs
    """
    dataframes = []
    for table_name in table_names:
        df = spark.table(f"{schema}.{table_name}")
        dataframes.append(df)
    return dataframes

In [0]:
# Deduplica as colunas com mesmo nome
def deduplicate_columns(df: DataFrame) -> DataFrame:
    columns = df.columns
    renamed_columns = [f"{col}@{i}".upper() for i, col in enumerate(columns)]
    
    # Rename columns to make them unique
    df_renamed = df.toDF(*renamed_columns)
    
    # Identify and keep the first occurrence of each column
    unique_columns = []
    seen = set()
    for i, col in enumerate(columns):
        if col not in seen:
            unique_columns.append(renamed_columns[i])
            seen.add(col)
    
    # Select only the unique columns
    df_unique = df_renamed.select(*unique_columns)
    
    # Optionally, rename columns back to original names, if needed
    original_names = [col.split('@')[0] for col in unique_columns]
    df_final = df_unique.toDF(*original_names)
    
    return df_final

In [0]:
# Teste deduplicação de colunas
# Example DataFrame with duplicate column names
data = [(1, 'A', 100), (2, 'B', 200)]
columns = ['id', 'name', 'id']  # Duplicate 'id' column

df = spark.createDataFrame(data, columns)

# Show initial DataFrame
print("Initial DataFrame with duplicate columns:")
df.show()

# Deduplicate columns
df_deduplicated = deduplicate_columns(df)

# Show DataFrame after deduplicating columns
print("DataFrame after deduplicating columns:")
df_deduplicated.show()

Initial DataFrame with duplicate columns:
+---+----+---+
| id|name| id|
+---+----+---+
|  1|   A|100|
|  2|   B|200|
+---+----+---+

DataFrame after deduplicating columns:
+---+----+
| ID|NAME|
+---+----+
|  1|   A|
|  2|   B|
+---+----+



### 2 - Execução

In [0]:
# %sql
# DROP TABLE databox.juridico_comum.tb_fechamento_civel_202406

In [0]:

# Lista todas as tabelas no catálogo com o nome buscado
tables = spark.sql(f"SHOW TABLES IN {nome_schema}").collect()
tables = spark.createDataFrame(tables)

tables = tables.where(f"tableName LIKE {nome_tabelas}")

table_names = [row.tableName for row in tables.select('tableName').collect()]

table_names.sort(reverse = True)

print(table_names)

['tb_fechamento_civel_202511', 'tb_fechamento_civel_202510', 'tb_fechamento_civel_202508', 'tb_fechamento_civel_202501', 'tb_fechamento_civel_202412', 'tb_fechamento_civel_202411', 'tb_fechamento_civel_202410', 'tb_fechamento_civel_202409', 'tb_fechamento_civel_202408', 'tb_fechamento_civel_202407', 'tb_fechamento_civel_202406', 'tb_fechamento_civel_202405', 'tb_fechamento_civel_202404', 'tb_fechamento_civel_202403', 'tb_fechamento_civel_202402', 'tb_fechamento_civel_202401', 'tb_fechamento_civel_202312', 'tb_fechamento_civel_202311', 'tb_fechamento_civel_202310', 'tb_fechamento_civel_202309', 'tb_fechamento_civel_202308', 'tb_fechamento_civel_202307', 'tb_fechamento_civel_202306', 'tb_fechamento_civel_202305', 'tb_fechamento_civel_202304', 'tb_fechamento_civel_202303', 'tb_fechamento_civel_202302', 'tb_fechamento_civel_202301', 'tb_fechamento_civel_202212', 'tb_fechamento_civel_202211', 'tb_fechamento_civel_202210', 'tb_fechamento_civel_202209', 'tb_fechamento_civel_202208', 'tb_fecha

#### 2.1 Nomes das colunas

In [0]:
lista_dfs = load_dataframes(nome_schema, table_names)

# Ajusta os nomes de colunas para remover exesso de espaços e "_"
lista_dfs_clean = []
for df in lista_dfs:
    df_clean = adjust_column_names(df)
    df_clean = remove_acentos(df_clean)
    df_clean = deduplica_cols(df_clean)
    lista_dfs_clean.append(df_clean)
   

In [0]:
# Renomear coluna No_PROCESSO para NO_PROCESSO em todos os DataFrames
lista_dfs_renomeados = []
for df in lista_dfs_clean:
    if "No_PROCESSO" in df.columns:
        df_renomeado = df.withColumnRenamed("No_PROCESSO", "NO_PROCESSO")
    else:
        df_renomeado = df
    lista_dfs_renomeados.append(df_renomeado)

In [0]:
df_clean = df_clean.withColumnRenamed("No_PROCESSO", "NO_PROCESSO")

#### 2.2 Deduplica

In [0]:
# Deduplica colunas com mesmo nome
lista_dfs_clean_dd = []
for df in lista_dfs_clean:
    df_dd = deduplicate_columns(df)
    lista_dfs_clean_dd.append(df_dd)

#### 2.3 Cast string

In [0]:
# Cast string todas as colunas
lista_dfs_clean_dd_string = []
for df in lista_dfs_clean_dd:
    for col in df.columns:
        df = df.withColumn(col, df[col].cast(StringType()))
    
    lista_dfs_clean_dd_string.append(df)

# lista_dfs_clean_dd_string[0].printSchema()

#### 2.4 All schema

In [0]:
def get_all_schema(dfs: list[DataFrame]) -> StructType:
    """
    Pega a lista de dfs e retorna o schema em comum. Ou seja,
    todas as colunas presentes em todas as tabelas.
    """
    # Get the schemas of all DataFrames
    schemas = [df.schema.fields for df in dfs]
    
    # Flatten the list of fields and use a set to find unique fields
    all_fields = {field for schema in schemas for field in schema}
    
    # Create a StructType with all the unique fields
    all_schema = StructType(list(all_fields))
    
    return all_schema

In [0]:
def create_empty_df_with_all_schema(dfs: list[DataFrame]) -> DataFrame:
    """"
    Usa as colunas em comum para criar um DF vazio.
    """
    # Get the all-inclusive schema
    all_schema = get_all_schema(dfs)
    
    # Create an empty DataFrame with the all-inclusive schema
    empty_df = spark.createDataFrame([], all_schema)
    
    return empty_df

In [0]:
unified_df = create_empty_df_with_all_schema(lista_dfs_clean_dd_string)

In [0]:
# Ensure unified_df does not already contai
unified_df = unified_df.withColumnRenamed('no_processo', 'NO_PROCESSO')

### 3 - Union

In [0]:
for df in lista_dfs_clean_dd_string:
    unified_df = unified_df.unionByName(df, allowMissingColumns=True)

# unified_df.display()

# To display the unified DataFrame
# display(unified_df)

### 4 - Recast

In [0]:
# Cria um DF com seus tipos originais
df_ref = create_empty_df_with_all_schema(lista_dfs_clean_dd)

# Usa esse schema como referencia para converter novamente os tipos do DF unido
schema_ref = df_ref.schema

for field in schema_ref.fields:
    unified_df = unified_df.withColumn(field.name, unified_df[field.name].cast(field.dataType))

In [0]:
unified_df.count()

1536137

### 5 - Salva a tabela final


In [0]:
#spark.sql("DROP TABLE databox.juridico_comum.fechamento_civel_consolidado")

In [0]:
unified_df.write.format("delta").option("overwriteSchema", "true").mode("overwrite").saveAsTable(f"{nome_schema}.{nome_tabela_unida}")

In [0]:
display(unified_df.groupBy("ESTRATEGIA").agg({"MES_FECH": "count", "ESTRATEGIA": "sum"}))

ESTRATEGIA,count(MES_FECH),sum(ESTRATEGIA)
ACORDO,552229,null
null,24057,null
DEFESA,871321,null
NaN,82716,NaN
-,1352,null
VIA VAREJO,1,null
DEFES,2564,null
ACORD,1872,null
0,25,0.0


In [0]:
unified_df.count()

1536137

In [0]:
dbutils.notebook.exit("Success")